# Потok API — исследование структуры GET /jobs.json
Собираем 500 свежих вакансий, смотрим примеры значений по каждому полю и выводим структуру для анализа.

In [ ]:
import requests
import pandas as pd
import json
from IPython.display import display

# ─── НАСТРОЙКИ ────────────────────────────────────────────────────────────────
API_TOKEN  = 'ВАШ_ТОКЕН_ЗДЕСЬ'
BASE_URL   = 'https://app.potok.io/api/v3'
TARGET     = 500   # сколько записей собрать
PER_PAGE   = 100   # максимум на страницу
BY_SCOPE   = 'all' # 'all' | 'active'
# ──────────────────────────────────────────────────────────────────────────────

HEADERS = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type':  'application/json',
}

In [ ]:
# ─── СБОР ДАННЫХ (постраничный) ───────────────────────────────────────────────
records = []
page    = 1

while len(records) < TARGET:
    need   = min(PER_PAGE, TARGET - len(records))
    params = {'by_scope': BY_SCOPE, 'page': page, 'per_page': need}
    resp   = requests.get(f'{BASE_URL}/jobs.json', headers=HEADERS, params=params)

    if resp.status_code != 200:
        print(f'Ошибка {resp.status_code}: {resp.text[:300]}')
        break

    batch = resp.json().get('data', [])
    if not batch:
        print(f'Пустой ответ на странице {page} — данные закончились.')
        break

    records.extend(batch)
    print(f'  стр. {page}: получено {len(batch)}, итого {len(records)}')
    page += 1

print(f'\nВсего собрано: {len(records)} записей')

In [ ]:
# ─── УТИЛИТЫ ──────────────────────────────────────────────────────────────────

def get_type_label(value):
    """Человекочитаемое название типа значения."""
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'bool'
    if isinstance(value, int):
        return 'int'
    if isinstance(value, float):
        return 'float'
    if isinstance(value, str):
        return 'str'
    if isinstance(value, list):
        if not value:
            return 'list[]'
        inner = get_type_label(value[0])
        return f'list[{inner}]'
    if isinstance(value, dict):
        return 'dict{' + ', '.join(value.keys()) + '}'
    return type(value).__name__


def collect_examples(records, field, n=3):
    """Собирает до n уникальных непустых примеров значений поля."""
    seen, examples = set(), []
    for rec in records:
        val = rec.get(field)
        if val is None:
            continue
        key = json.dumps(val, ensure_ascii=False, default=str)
        if key not in seen:
            seen.add(key)
            examples.append(val)
        if len(examples) >= n:
            break
    return examples


def infer_schema(value, _depth=0):
    """
    Рекурсивно строит схему (без данных) для произвольного значения.
    """
    indent = '  ' * _depth
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'bool'
    if isinstance(value, int):
        return 'int'
    if isinstance(value, float):
        return 'float'
    if isinstance(value, str):
        return 'str'
    if isinstance(value, list):
        if not value:
            return 'list[]'
        item_schema = infer_schema(value[0], _depth + 1)
        return f'list[\n{indent}  {item_schema}\n{indent}]'
    if isinstance(value, dict):
        lines = []
        for k, v in value.items():
            lines.append(f'{indent}  {k}: {infer_schema(v, _depth + 1)}')
        inner = '\n'.join(lines)
        return f'dict{{\n{inner}\n{indent}}}'
    return type(value).__name__


print('Утилиты загружены.')

In [ ]:
# ─── СБОР ВСЕХ КЛЮЧЕЙ ─────────────────────────────────────────────────────────
all_keys = set()
for rec in records:
    all_keys.update(rec.keys())

# Сортируем: сначала скалярные поля, потом объекты/массивы
def sort_key(k):
    sample = next((r.get(k) for r in records if r.get(k) is not None), None)
    return (0 if not isinstance(sample, (dict, list)) else 1, k)

sorted_keys = sorted(all_keys, key=sort_key)
print(f'Всего уникальных полей: {len(sorted_keys)}')
print('Поля:', sorted_keys)

In [ ]:
# ─── ТАБЛИЦА: ПРИМЕРЫ ЗНАЧЕНИЙ ────────────────────────────────────────────────
# Для каждого поля: тип, заполненность, 2-3 примера непустых значений

def fmt(v, maxlen=120):
    s = json.dumps(v, ensure_ascii=False, default=str)
    return s[:maxlen] + '…' if len(s) > maxlen else s

rows = []
for key in sorted_keys:
    non_null  = sum(1 for r in records if r.get(key) is not None)
    fill_pct  = non_null / len(records) * 100 if records else 0
    sample    = next((r.get(key) for r in records if r.get(key) is not None), None)
    examples  = collect_examples(records, key, n=3)

    rows.append({
        'field':     key,
        'type':      get_type_label(sample),
        'fill_%':    f'{fill_pct:.0f}%',
        'example_1': fmt(examples[0]) if len(examples) > 0 else '—',
        'example_2': fmt(examples[1]) if len(examples) > 1 else '—',
        'example_3': fmt(examples[2]) if len(examples) > 2 else '—',
    })

df_examples = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_rows', 200)
display(df_examples)

In [ ]:
# ─── СХЕМА СТРУКТУРЫ (без данных) ─────────────────────────────────────────────
# Этот вывод можно скопировать и отправить на анализ

schema_lines = []
for key in sorted_keys:
    sample = next((r.get(key) for r in records if r.get(key) is not None), None)
    schema_lines.append(f'{key}: {infer_schema(sample)}')

schema_str = '\n'.join(schema_lines)
print('=' * 70)
print('SCHEMA GET /jobs.json — скопируй и отправь на анализ:')
print('=' * 70)
print(schema_str)

In [ ]:
# ─── ДЕТАЛИ ВЛОЖЕННЫХ ПОЛЕЙ ───────────────────────────────────────────────────
# Разворачиваем каждое dict/list-поле отдельно + живой пример

nested_fields = [
    k for k in sorted_keys
    if isinstance(next((r.get(k) for r in records if r.get(k) is not None), None), (dict, list))
]

print(f'Вложенные поля ({len(nested_fields)}): {nested_fields}\n')
print('=' * 70)

for field in nested_fields:
    sample = next((r.get(field) for r in records if r.get(field) is not None), None)
    print(f'\n── {field} ──')
    print('Схема:')
    print(infer_schema(sample))
    raw = json.dumps(sample, ensure_ascii=False, indent=2, default=str)
    if len(raw) > 600:
        raw = raw[:600] + '\n  ... (обрезано)'
    print('\nПример значения:')
    print(raw)
    print()

In [ ]:
# ─── ЭКСПОРТ СХЕМЫ В ФАЙЛ ─────────────────────────────────────────────────────

with open('jobs_schema.txt', 'w', encoding='utf-8') as f:
    f.write('=== SCHEMA GET /jobs.json ===\n\n')
    f.write(schema_str)
    f.write('\n\n=== NESTED FIELD DETAILS ===\n')
    for field in nested_fields:
        sample = next((r.get(field) for r in records if r.get(field) is not None), None)
        f.write(f'\n── {field} ──\n')
        f.write(infer_schema(sample))
        f.write('\n')

print('Схема сохранена в jobs_schema.txt')